In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

<font size="5" color="black">ch9. RAG</font>

# 1. knowledge base 구성을 위한 데이터 생성

In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = Docx2txtLoader('./tax/소득세법(법률)(제21221호)(20260701).docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    separators=["\n\n\n\n", "\n\n", "\n", " "] # 분할 기준 우선순위
)

documents = loader.load_and_split(text_splitter=text_splitter)

C:\Users\Admin\AppData\Local\Temp\ipykernel_7512\1660385314.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [3]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

import os
load_dotenv()

embedding = OpenAIEmbeddings(
    model='text-embedding-3-large'
)

In [4]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
pc = Pinecone()

C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# %%time
# 데이터 첫 저장 시
# index_name = 'tax-index-openai'
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )

CPU times: total: 6.27 s
Wall time: 14.1 s


In [6]:
%%time
# vectorDB 로드 시
index_name = 'tax-index-openai'
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 0 ns
Wall time: 2.74 ms


# 2. 답변 생성을 위한 Retrieval

In [7]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(
    query=query, 
    k=2 # 기본값 : 4
)

In [8]:
retrieved_doc = "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [9]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model='gpt-4.1-nano')

In [10]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세법 전문가 입니다
- 다음의 [context]를 참조하여 사용자의 질문에 5문장 이내로 답변하세요
[context]
{retrieved_doc}
질문 : {query}"""

ai_msg = llm.invoke(prompt)

# 3. RAG chain

In [12]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

# 임베딩 & LLM 초기화
load_dotenv()
embedding = OpenAIEmbeddings(model = 'text-embedding-3-large')
llm = ChatOpenAI(model = 'gpt-4.1-nano')

index_name = 'tax-index-openai'
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

# Retriever 생성
retriever = database.as_retriever(
    search_type='similarity',
    search_kwargs={'k':3}
)

# 프롬프트 템플릿
template = f'''당신은 최고의 한국 소득세법 전문가입니다. 다음 문맥을 참고하여 질문에 최대 5문장으로 답하세요. 답을 모르면 모른다고 답하세요.
질문 : {{query}}
문맥 : {{context}}
답변 : '''
prompt = ChatPromptTemplate.from_template(template)

# 검색한 문서 텍스트 변환
def format_documents(retrieved_docs):
    return "\n--\n".join([doc.page_content.replace('\n\n', ' ') for doc in retrieved_docs])

In [13]:
# RAG chain
rag_chain = (
    {
        "context":retriever | format_documents,
        "query":RunnablePassthrough() # 질문만 그대로 전달
    }
    |prompt
    |llm
    |StrOutputParser()
)

rag_chain.invoke('연봉 5천만원인 직장인의 소득세는 얼만가요?')

'연봉 5천만원인 직장인의 소득세는 정확한 계산을 위해 여러 요소를 고려해야 하지만, 일반적으로 근로소득공제와 기타 공제 후 과세표준에 따라 세율이 적용됩니다. 2023년 기준으로 기본 공제, 인적공제, 근로소득공제 등을 차감하면, 과세표준은 약 3,500만원에서 4,000만원 수준이 될 수 있습니다. 이에 대한 소득세율은 누진세 구조에 따라 약 6%에서 15% 내외로 계산됩니다. 예를 들어, 근로소득세율 구간이 약 8% 내외로 추정되어 연간 세금은 약 300만원에서 400만원 정도일 가능성이 높습니다. 정확한 세액 산정을 위해서는 세법상 공제 항목과 세율 적용 범위를 상세히 계산해야 합니다.'